In [20]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT


conn = psycopg2.connect(
    "host=192.168.10.77 port=5432 user=shut password=shut dbname=shut_archive")
#conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

In [19]:
conn.close()

In [21]:
cur = conn.cursor()

In [ ]:
schema_name = 'pd'
cur.execute(f"DROP SCHEMA IF EXISTS {schema_name}")
conn.commit()
print("schema dropped")

In [ ]:
query = '''
SELECT table_name FROM information_schema.tables WHERE table_schema = 'pd'
'''

cur.execute(query)
result = cur.fetchall()
print(result)
cur.close()
conn.close()

In [ ]:
schema_name = 'pd'
cur.execute(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
conn.commit()
print("schema created")

In [ ]:
import os

csv_directory = r'C:\Users\Nabil\Desktop\Softia_Work_Repos\SHUTRefonte\shut-refonte\ScriptLectureReleves\Voies_PD_TOR'
for filename in os.listdir(csv_directory):
    if filename.endswith(".csv"):
        table_name = f"{os.path.splitext(filename)[0]}".lower()
        print(table_name)
        cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {schema_name}.{table_name} (
            horodate TIMESTAMPTZ NOT NULL,
            evenement TEXT,
            mesure DOUBLE PRECISION,
            mesure_brute TEXT
        )
        """)
        cur.execute(
            f"SELECT create_hypertable('{schema_name}.{table_name}', 'horodate', if_not_exists => TRUE)")
conn.commit()

In [ ]:
import pandas as pd
import datetime
import os


csv_directory = r'C:\Users\Nabil\Desktop\Softia_Work_Repos\SHUTRefonte\shut-refonte\ScriptLectureReleves\Voies_PD_TOR'


schema_name = 'pd'


for filename in os.listdir(csv_directory):
    print(filename)
    csv_file = os.path.join(csv_directory, filename)
    table_name = f"{os.path.splitext(filename)[0]}".lower()
    print(table_name)
    print(csv_file)
    df = pd.read_csv(csv_file, parse_dates=['horodate'])
    count = 0
    with conn.cursor() as cur:
        for _, row in df.iterrows():
            query = f"""
            INSERT INTO {schema_name}.{table_name} (horodate, evenement, mesure, mesure_brute)
            VALUES ('{row['horodate']}', '{row['evenement']}', {row['mesure']}, '{row['mesure_brute']}') 
            """
            cur.execute(query)
            count += 1
            if(count % 100000 == 0):
                print(count)
                conn.commit()
        conn.commit()

In [ ]:

cur = conn.cursor()
cur.execute(
    "SELECT * FROM information_schema.tables WHERE table_schema = 'pd'")
result = cur.fetchall()
print(result)

In [ ]:
stations = list(map(lambda x: x[2], result))
print(stations)

In [ ]:
cur.execute(
    "SELECT schema_name FROM information_schema.schemata;")
result = cur.fetchall()
print(result)

In [ ]:
list_schema_pg = ['pg_catalog', 'public', 'information_schema','timescaledb_experimental', 'timescaledb_information','_timescaledb_internal', '_timescaledb_cache', '_timescaledb_config', '_timescaledb_functions', '_timescaledb_catalog']
stations = list(map(lambda x: x[0] if not (str(x[0]) in list_schema_pg) else None, result))
stations_reduc = list(filter(lambda x: x is not None, stations))
print(stations_reduc)

In [ ]:


query = '''
SELECT min(horodate), max(horodate) FROM pd.qco_vmer_pd

'''
#print(query)
cur.execute(query)
res_table = cur.fetchall()
print(res_table)

In [ ]:
cur.close()
conn.close()

In [ ]:
#TODO container influx same same